# YouTube Trending Music Metadata and Relative Engagement

**Authors:** *Andranik Pambukhchyan, Asilbek Amonov*  
**Course:** SOSC 314  
**Repository:** https://github.com/Andran1k/SOSC314_CourseProject

This notebook is the source for the public-facing final report. The final analysis is restricted to the four English-speaking regions `US`, `GB`, `CA`, and `AU` so that language-based interpretation is not mixed across different languages.


In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root()
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "trending_music_processed.csv")
diag = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "diagnostics_week5_results_cv.csv")
title_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "final_report_title_description_results.csv")
timing_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "final_report_timing_results.csv")
release_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "final_report_release_labels.csv")
structure_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "final_report_structure_results.csv")

df.shape


## Research Question and Motivation

This project asks whether the language used in YouTube trending music-video metadata helps predict which videos receive relatively higher engagement within the trending set. We focus on titles and descriptions because they are public, visible, and intentionally written. They can contain artist names, collaboration markers, release framing, and promotional cues that may matter for attention on the platform.

At the same time, this is not a causal project. We do not claim that changing a title or description would automatically raise views. View counts also depend on recommendation exposure, artist popularity, release strategy, thumbnails, timing, and label promotion. The goal is therefore more modest: to test whether metadata language is informative, how strong that information is, and what other non-text signals seem to matter in the same sample.

The final report extends the original project in four ways. First, it compares title-only, description-only, and combined text models. Second, it adds timing variables based on upload time and collection lag. Third, it codes a few transparent release-format labels from titles. Fourth, it compares text with simple promo and structure variables. Together, these additions make the project less dependent on one overall AUC number and give stronger substantive conclusions.


## Data

### Data source and scope

The data comes from the YouTube Data API v3 using the `videos.list` endpoint with `chart=mostPopular`. The raw files are stored in `data/raw/`. The final main dataset in this report is `data/processed/trending_music_processed.csv`, which is explicitly filtered to the four English-speaking regions `US`, `GB`, `CA`, and `AU`. A separate archive file, `trending_music_processed_all_regions.csv`, keeps the broader multi-region dataset, but it is not used for the final language-sensitive claims.

### Unit of analysis

The unit of analysis is one trending YouTube music video. Each row contains the video metadata, region, engagement counts, cleaned text fields, timing features, and a binary outcome `high_views`, coded as 1 if `view_count` is above the sample median and 0 otherwise.

### Text and timing features

The project uses two main text operationalizations. The first is promo-inclusive text, which combines the cleaned title and description. The second is semantic-filtered text, which removes common promotional phrases using the list in `assets/promo_phrases.txt`. In the extended analysis, we also create title-only and description-only versions, release-format indicators such as `has_official_video` and `has_lyrics`, and timing features derived from `published_at` and the first collection timestamp observed in the raw files.

### Why the four-region restriction matters

Earlier project files included more regions, but the final report uses only `US`, `GB`, `CA`, and `AU`. The reason is straightforward: we want the language analysis to stay interpretable. Mixing English and non-English metadata would make it harder to know whether the model is learning language, region, or translation differences.


In [ ]:
summary_df = pd.DataFrame(
    {
        "Statistic": [
            "Number of videos",
            "Regions",
            "Median view count",
            "Mean token count (promo)",
            "Mean token count (semantic)",
            "Mean lexical diversity (promo)",
            "Mean lag to first collection (hours)",
        ],
        "Value": [
            len(df),
            ", ".join(sorted(df['region'].unique().tolist())),
            int(df['view_count'].median()),
            round(df['token_count_promo'].mean(), 1),
            round(df['token_count_semantic'].mean(), 1),
            round(df['lexdiv_promo'].mean(), 3),
            round(df['lag_hours_first_seen'].mean(), 1),
        ],
    }
)
summary_df


The final dataset contains 136 videos across four regions. This is enough for a course project, but it is still a moderate sample, which is why the report relies on repeated cross-validation instead of one lucky train/test split.


## Problem Setup and Methods

The main task is binary text classification: can we predict whether a trending music video is above the sample median in views? The report keeps a simple baseline and a simple improved model. The baseline is a majority-class dummy classifier. The main trained models use bag-of-words features with logistic regression, evaluated through repeated stratified cross-validation.

The project keeps the earlier promo-versus-semantic diagnostic, but it now adds four stronger extensions:

1. title-only vs description-only vs combined text,
2. timing and lag-to-collection analysis,
3. release-format indicators from titles,
4. text vs promo/structure comparisons.

These additions are important because they let us move from a generic statement like "text matters a bit" to a more precise statement about what kind of signal the model is using.


In [ ]:
main_results = diag[diag['diagnostic'].isin(['sanity_baseline', 'vectorizer'])].copy()
main_results[['config', 'auc_mean', 'auc_sd', 'acc_mean', 'acc_sd', 'n_folds']]


## Baseline Evaluation

The baseline still does what we want it to do: it stays at chance. The best main text model remains TF-IDF with logistic regression, which reaches an average AUC of about **0.694** on the four-region sample. That is a modest result rather than a strong one, but it is enough to say that metadata language contains some useful information.

The earlier promo-versus-semantic comparison also still holds. Using Count features, promo and semantic text perform almost identically (about **0.634** AUC either way). This means semantic filtering is useful for interpretation, but it should not be presented as a major predictive improvement.


![Week 5 diagnostics (four-region sample)](../figures/diagnostics_week5_auc_cv.png)

*Figure 1. Cross-validated diagnostics on the English-only sample. Promo and semantic text perform similarly, while TF-IDF improves over simple count features.*


## New Finding 1: Where is the text signal?

The first extension asks whether the predictive signal is really in the titles, the descriptions, or both together. A reasonable expectation might be that titles dominate because they are short, visible, and heavily branded. However, the four-region results are more interesting than that.


In [ ]:
title_results


The combined model performs best at **0.689** AUC, but **description-only text reaches 0.685**, which is very close. **Title-only text is weaker at 0.651**. This is one of the most important new results in the project. It suggests that the long description field, even though it often contains promotional or boilerplate material, still carries a meaningful amount of signal.

In other words, the project should not conclude that descriptions are just noise. In this sample they appear to matter almost as much as the title itself, and the combined text performs only slightly better than descriptions alone.


![Title vs description vs combined text](../figures/final_title_description_auc.png)

*Figure 2. Description-only text performs surprisingly close to the combined model, while title-only text is weaker.*


## New Finding 2: Timing matters at least as much as text

The second extension adds upload-time and collection-lag features. Here the result is even stronger than expected. A timing-and-structure-only model reaches about **0.724** AUC, which is higher than the **0.689** AUC from text alone. The combined text-plus-timing model improves slightly again to **0.728**.

This does not mean text is irrelevant. It means that timing and packaging context are very important in this sample. Once we add features like upload weekday, upload hour, lag to first collection, and simple document-length measures, we can predict above-median views better than with text alone.

Interestingly, the lag result is not in the expected direction. Above-median videos have a longer mean lag to first collection in our data than below-median videos. That suggests the lag variable may be capturing release-cycle patterns rather than simple "faster trending" behavior. We should therefore treat lag as informative but not interpret it as a direct measure of how quickly a video truly entered YouTube's trending system.


In [ ]:
timing_results[timing_results['section'] == 'model_comparison'][['model', 'auc_mean', 'auc_sd', 'acc_mean', 'acc_sd']]


![Timing patterns in the four-region sample](../figures/final_timing_patterns.png)

*Figure 3. Timing variables are informative, but the weekday pattern should be treated descriptively because some days have small cell counts.*


## New Finding 3: Packaging labels matter, but unevenly

The third extension codes a few simple rule-based title labels: `official video`, `lyrics / lyric video`, and collaboration markers such as `feat` or `ft`. These labels are not perfect, but they are transparent and easy to explain.


In [ ]:
release_results


Two patterns stand out. Videos labeled as **official video** are above the median about **65%** of the time, compared with about **47%** for the rest of the sample. **Lyrics / lyric video** titles are above the median about **69%** of the time, although the count is small and should be treated carefully. Collaboration markers are weaker: they are only slightly above the overall baseline.

This part of the project helps the final conclusion because it turns an abstract modeling result into a more concrete claim. The model is not only learning general language. It is also picking up familiar music-release packaging conventions.


![Release labels and high-view rate](../figures/final_release_type_high_views.png)

*Figure 4. Official-video and lyric-style labels are associated with higher relative engagement in this sample, while collaboration markers are weaker.*


## New Finding 4: Promo and structure carry signal, but mostly overlapping signal

The final extension compares text with a small set of promo and structure variables such as `promo_share`, title length, description length, and lexical diversity. Here the results are more mixed.


In [ ]:
structure_results[structure_results['section'] == 'model_comparison'][['model', 'auc_mean', 'auc_sd', 'acc_mean', 'acc_sd']]


Promo and structure variables by themselves reach about **0.659** AUC, so they are not useless. However, when they are combined with text, the result does **not** improve beyond the text-only model. In fact, the combined AUC is slightly lower in this run. That means these variables are probably overlapping with information already present in the text rather than adding a new independent signal.

The promo-share quartile pattern also suggests that the lowest-promo documents perform worst, but the relationship is not monotonic enough to support a strong substantive claim. The safest reading is that packaging structure matters, but not in a simple "more promo equals more views" way.


![Promo and structure signal](../figures/final_promo_structure_signal.png)

*Figure 5. Promo and structure variables carry some signal, but they do not clearly improve prediction once text is already included.*


## Interpretation and Final Conclusion

The original conclusion of the project still holds, but it now needs to be stated more precisely. Metadata language does contain predictive information about relative engagement among trending music videos in `US`, `GB`, `CA`, and `AU`. A text model performs better than chance, and TF-IDF remains the strongest main text representation in the report.

However, the extensions show that text is only one part of the story. First, descriptions carry almost as much signal as the combined title-and-description field, while titles alone are weaker. Second, timing and simple structural variables matter at least as much as text and slightly outperform text-only models in this sample. Third, packaging labels such as `official video` and `lyrics` are associated with higher relative engagement. Fourth, promo and structure features overlap with text rather than clearly adding an independent performance gain.

Putting these pieces together, the strongest final conclusion is this: **relative engagement among trending music videos is associated with a mixture of language, timing, and packaging signals rather than a pure effect of wording alone.** The project therefore ends with a stronger and more defensible claim than before. The model is not simply learning catchy phrases. It is learning how releases are presented, when they appear, and which kinds of metadata conventions travel with higher-view videos in this English-language trending sample.

Just as importantly, the project still does not support a causal interpretation. It does not show that rewriting a title would increase views, and it does not generalize automatically beyond the trending music context. What it does show is that visible metadata and basic release context together provide a modest but meaningful signal of relative performance.


## Limitations and Future Work

This project still has important limits:

1. It studies only trending music videos, not all YouTube uploads.
2. The final report is restricted to four English-speaking regions for interpretability, so the conclusion is not a global YouTube claim.
3. `published_at` timing is in UTC, which may not perfectly match local release strategy.
4. `lag_hours_first_seen` is based on the first time a video appears in our collected raw files, not the true internal YouTube trending timestamp.
5. The release labels are rule-based and transparent, but they are still imperfect measurements.
6. The sample is moderate in size, so some descriptive breakdowns, especially weekday and release-label subsets, should be interpreted cautiously.

If we continued this project, the next steps would be to collect a longer time window, improve the timing measurements, separate title language from boilerplate description blocks more carefully, and compare the same framework with one or two non-music categories.


## References

- YouTube Data API v3 documentation for `videos.list` and `chart=mostPopular`.
- pandas documentation.
- matplotlib documentation.

**Generative AI acknowledgment:** Generative AI tools were used for drafting, restructuring, and code assistance during project development. Final claims and report text were checked against the repository outputs.
